# 面试问题：Knowledge Distillation 如何实现，温度和损失权重分别起什么作用？

可以直接复述的回答是：知识蒸馏让小模型同时学习硬标签与教师模型的软分布。高温 softmax 会暴露非目标类别之间的相对关系，也就是常说的 dark knowledge。学生通常优化 `alpha * soft_loss + (1-alpha) * hard_loss`，其中 soft loss 要乘 `T²`，补偿 softmax 对 logits 梯度随温度缩小的效应。教师必须处于 eval/no-grad 状态，学生才执行 backward 和参数更新。蒸馏是否成功不能只看分类准确率，还要看学生与教师分布的 KL、延迟和模型大小。下面用十张工单手写温度 softmax、软交叉熵和学生训练。

## 真实案例：把工单优先级教师压缩成线性学生模型

输入字段是紧急程度、系统中断、VIP 和金额损失，标签为普通、加急、严重。十条记录是结构与客服系统一致的脱敏教学数据；教师规则经过人为构造，小样本结果不代表线上泛化。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量与自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警以保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(10)  # 固定学生初始化
label_names = ["普通", "加急", "严重"]  # 定义三个优先级可读名称
records = [  # 定义十张带业务语义的工单
    ("K-01", 0.10, 0.0, 0.0, 0.00, 0),  # 低紧急普通咨询
    ("K-02", 0.30, 0.0, 1.0, 0.10, 1),  # VIP 轻度损失工单
    ("K-03", 0.80, 0.0, 0.0, 0.20, 1),  # 高紧急但未中断工单
    ("K-04", 0.60, 1.0, 0.0, 0.80, 2),  # 中断且损失较高工单
    ("K-05", 0.40, 0.0, 0.0, 0.00, 0),  # 普通咨询边界样本
    ("K-06", 0.90, 1.0, 1.0, 1.00, 2),  # VIP 全面中断严重样本
    ("K-07", 0.70, 0.0, 1.0, 0.40, 1),  # VIP 高紧急加急样本
    ("K-08", 0.20, 1.0, 0.0, 0.70, 2),  # 紧急度低但系统中断样本
    ("K-09", 0.15, 0.0, 0.0, 0.05, 0),  # 稳定普通样本
    ("K-10", 0.50, 0.0, 1.0, 0.20, 1),  # VIP 中等紧急样本
]  # 结束十张工单
x = torch.tensor([[row[1], row[2], row[3], row[4]] for row in records], dtype=torch.float32)  # 构造四维工单特征
y = torch.tensor([row[5] for row in records], dtype=torch.long)  # 构造三分类硬标签
print("输入预览：id | 紧急度 | 中断 | VIP | 损失 | 标签")  # 输出原始字段标题
for row in records:  # 逐条展示十张工单
    print(f"{row[0]} | {row[1]:.2f} | {int(row[2])} | {int(row[3])} | {row[4]:.2f} | {label_names[row[5]]}")  # 展示业务输入和硬标签
print("训练张量形状：", tuple(x.shape), tuple(y.shape))  # 展示学生模型输入输出规模

输入预览：id | 紧急度 | 中断 | VIP | 损失 | 标签
K-01 | 0.10 | 0 | 0 | 0.00 | 普通
K-02 | 0.30 | 0 | 1 | 0.10 | 加急
K-03 | 0.80 | 0 | 0 | 0.20 | 加急
K-04 | 0.60 | 1 | 0 | 0.80 | 严重
K-05 | 0.40 | 0 | 0 | 0.00 | 普通
K-06 | 0.90 | 1 | 1 | 1.00 | 严重
K-07 | 0.70 | 0 | 1 | 0.40 | 加急
K-08 | 0.20 | 1 | 0 | 0.70 | 严重
K-09 | 0.15 | 0 | 0 | 0.05 | 普通
K-10 | 0.50 | 0 | 1 | 0.20 | 加急
训练张量形状： (10, 4) (10,)


## Baseline / 基线：学生只学习 one-hot 硬标签

教师对相邻等级保留概率，例如“加急”样本仍可能有少量“严重”概率。硬标签基线把这些相对关系全部丢掉。

In [2]:
class PriorityTeacher(torch.nn.Module):  # 定义较复杂的固定教师决策函数
    def forward(self, features):  # 根据四个业务字段生成三类 logits
        urgency = features[:, 0]  # 读取紧急程度特征
        outage = features[:, 1]  # 读取系统中断标记
        vip = features[:, 2]  # 读取 VIP 标记
        loss = features[:, 3]  # 读取归一化金额损失
        normal_logit = 2.5 - 2.0 * urgency - 1.0 * outage - 0.5 * vip - 1.0 * loss  # 计算普通类别教师分数
        urgent_logit = -0.2 + 2.0 * urgency - 1.0 * outage + 1.0 * vip + 0.5 * loss  # 计算加急类别教师分数
        critical_logit = -1.0 + 0.5 * urgency + 3.0 * outage + 1.5 * loss  # 计算严重类别教师分数
        return torch.stack([normal_logit, urgent_logit, critical_logit], dim=1)  # 拼接三类教师 logits
class LinearStudent(torch.nn.Module):  # 定义待蒸馏的轻量线性学生
    def __init__(self):  # 初始化学生权重和偏置
        super().__init__()  # 初始化 PyTorch 模块基类
        self.weight = torch.nn.Parameter(torch.randn(4, 3) * 0.05)  # 创建四特征到三类别权重
        self.bias = torch.nn.Parameter(torch.zeros(3))  # 创建三个类别偏置
    def forward(self, features):  # 定义学生前向传播
        return features @ self.weight + self.bias  # 计算线性分类 logits
def stable_probabilities(logits, temperature=1.0):  # 手写带温度的数值稳定 softmax
    scaled = logits / temperature  # 按温度缩放类别 logits
    shifted = scaled - scaled.max(dim=1, keepdim=True).values  # 减去行最大值防止指数溢出
    exponentials = torch.exp(shifted)  # 计算稳定指数分数
    return exponentials / exponentials.sum(dim=1, keepdim=True)  # 归一化为类别概率
def hard_cross_entropy(logits, labels):  # 手写基于 one-hot 的硬标签交叉熵
    probabilities = stable_probabilities(logits)  # 计算学生常温类别概率
    selected = probabilities[torch.arange(len(labels)), labels]  # 选出每条样本真实类别概率
    return -torch.log(selected + 1e-12).mean()  # 返回平均负对数似然
teacher = PriorityTeacher()  # 创建固定教师模型
with torch.no_grad():  # 禁止教师前向建立梯度图
    teacher_logits = teacher(x)  # 对十张工单生成教师 logits
    teacher_probabilities = stable_probabilities(teacher_logits)  # 计算教师常温概率供预览
initial_student = LinearStudent()  # 创建所有学生共享的初始化
initial_student_state = {name: value.detach().clone() for name, value in initial_student.state_dict().items()}  # 复制公平比较初始参数
def train_hard_student(steps=120, learning_rate=0.25):  # 用硬标签训练学生基线
    student = LinearStudent()  # 创建新的线性学生
    student.load_state_dict(initial_student_state)  # 恢复相同初始参数
    trace = []  # 保存关键训练步损失和梯度范数
    for step in range(1, steps + 1):  # 执行固定预算全批量训练
        logits = student(x)  # 真实执行学生 forward
        loss = hard_cross_entropy(logits, y)  # 只计算硬标签交叉熵
        loss.backward()  # 真实执行 backward 得到学生梯度
        gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in student.parameters()))  # 计算学生全局梯度范数
        with torch.no_grad():  # 关闭手写参数更新的计算图
            for parameter in student.parameters():  # 遍历学生权重和偏置
                parameter -= learning_rate * parameter.grad  # 使用简单 SGD 更新学生参数
                parameter.grad.zero_()  # 清空当前参数梯度
        if step in {1, 10, 40, steps}:  # 选择关键训练节点
            trace.append((step, float(loss), float(gradient_norm)))  # 保存硬标签训练轨迹
    return student, trace  # 返回训练后的学生和轨迹
hard_student, hard_trace = train_hard_student()  # 运行同数据硬标签基线
print("教师概率预览：id | P(普通, 加急, 严重)")  # 输出教师软标签表头
for index in range(5):  # 展示前五条教师分布
    print(f"{records[index][0]} | {[round(value, 3) for value in teacher_probabilities[index].tolist()]}")  # 展示非目标类别暗知识
print("硬标签训练：step | loss | grad_norm")  # 输出基线训练轨迹表头
for item in hard_trace:  # 遍历四个关键训练节点
    print(f"{item[0]:4d} | {item[1]:.5f} | {item[2]:.5f}")  # 展示真实反向传播和收敛过程

教师概率预览：id | P(普通, 加急, 严重)
K-01 | [0.878, 0.088, 0.034]
K-02 | [0.435, 0.506, 0.059]
K-03 | [0.278, 0.619, 0.102]
K-04 | [0.017, 0.042, 0.94]
K-05 | [0.707, 0.235, 0.058]
硬标签训练：step | loss | grad_norm
   1 | 1.11733 | 0.40715
  10 | 0.85769 | 0.28514
  40 | 0.51183 | 0.16102
 120 | 0.27046 | 0.07523


## 核心实现：温度软标签、T² 补偿和混合损失

教师 logits 只计算一次并 detach。学生在温度 3 下拟合完整教师分布，同时保留 25% 硬标签监督。

In [3]:
def soft_cross_entropy(student_logits, target_logits, temperature, compensate=True):  # 手写教师软标签交叉熵
    target_probability = stable_probabilities(target_logits.detach(), temperature)  # 生成不回传教师梯度的温度软标签
    student_probability = stable_probabilities(student_logits, temperature)  # 生成学生温度概率
    loss = -(target_probability * torch.log(student_probability + 1e-12)).sum(dim=1).mean()  # 计算完整类别分布交叉熵
    return loss * temperature ** 2 if compensate else loss  # 按需要应用 T² 梯度补偿
def train_distilled_student(steps=120, learning_rate=0.25, temperature=3.0, alpha=0.75):  # 训练软硬目标混合学生
    student = LinearStudent()  # 创建蒸馏学生模型
    student.load_state_dict(initial_student_state)  # 恢复与基线完全相同的初值
    trace = []  # 保存蒸馏训练中间过程
    for step in range(1, steps + 1):  # 执行与基线相同训练预算
        logits = student(x)  # 真实执行学生 forward
        hard_loss = hard_cross_entropy(logits, y)  # 计算硬标签监督损失
        soft_loss = soft_cross_entropy(logits, teacher_logits, temperature, True)  # 计算带 T² 的教师软损失
        total_loss = alpha * soft_loss + (1.0 - alpha) * hard_loss  # 按权重组合两种监督信号
        total_loss.backward()  # 真实执行混合目标 backward
        gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in student.parameters()))  # 计算当前学生梯度范数
        with torch.no_grad():  # 关闭参数更新的梯度记录
            for parameter in student.parameters():  # 遍历学生全部参数
                parameter -= learning_rate * parameter.grad  # 手写 SGD 更新蒸馏学生
                parameter.grad.zero_()  # 清空本步学生梯度
        if step in {1, 10, 40, steps}:  # 保存与基线相同的观察节点
            trace.append((step, float(hard_loss), float(soft_loss), float(total_loss), float(gradient_norm)))  # 保存软硬损失与梯度
    return student, trace  # 返回蒸馏学生和训练轨迹
distilled_student, distill_trace = train_distilled_student()  # 运行知识蒸馏主方案
print("蒸馏训练：step | hard | soft*T² | total | grad_norm")  # 输出蒸馏轨迹表头
for item in distill_trace:  # 遍历关键训练节点
    print(f"{item[0]:4d} | {item[1]:.5f} | {item[2]:.5f} | {item[3]:.5f} | {item[4]:.5f}")  # 展示两类监督如何共同变化

蒸馏训练：step | hard | soft*T² | total | grad_norm
   1 | 1.11733 | 9.90623 | 7.70900 | 0.41445
  10 | 0.85658 | 9.62334 | 7.43165 | 0.29875
  40 | 0.51401 | 9.23813 | 7.05710 | 0.16048
 120 | 0.31521 | 9.05526 | 6.87025 | 0.05153


## 逐样本结果：分类与教师分布距离同时比较

In [4]:
def mean_teacher_kl(student):  # 计算学生相对教师的平均 KL 散度
    student_probability = stable_probabilities(student(x))  # 获取学生常温预测分布
    teacher_probability = stable_probabilities(teacher_logits)  # 获取教师常温目标分布
    per_sample = (teacher_probability * (torch.log(teacher_probability + 1e-12) - torch.log(student_probability + 1e-12))).sum(dim=1)  # 逐样本计算 KL
    return float(per_sample.mean()), per_sample.detach(), student_probability.detach()  # 返回平均值、逐样本值和概率
hard_kl, hard_per_sample_kl, hard_probability = mean_teacher_kl(hard_student)  # 评估硬标签学生与教师差异
distilled_kl, distilled_per_sample_kl, distilled_probability = mean_teacher_kl(distilled_student)  # 评估蒸馏学生与教师差异
hard_prediction = hard_probability.argmax(dim=1)  # 生成硬标签学生类别预测
distilled_prediction = distilled_probability.argmax(dim=1)  # 生成蒸馏学生类别预测
hard_accuracy = float((hard_prediction == y).float().mean())  # 计算硬标签学生训练样本准确率
distilled_accuracy = float((distilled_prediction == y).float().mean())  # 计算蒸馏学生训练样本准确率
print("id | 标签 | 教师预测 | 硬学生 | 蒸馏学生 | KL硬 | KL蒸馏")  # 输出逐样本对照表头
for index, record in enumerate(records):  # 遍历十张工单
    teacher_prediction = int(teacher_probabilities[index].argmax())  # 获取教师当前预测类别
    print(f"{record[0]} | {label_names[int(y[index])]} | {label_names[teacher_prediction]} | {label_names[int(hard_prediction[index])]} | {label_names[int(distilled_prediction[index])]} | {hard_per_sample_kl[index]:.4f} | {distilled_per_sample_kl[index]:.4f}")  # 展示分类和分布拟合差异
print(f"硬标签：accuracy={hard_accuracy:.1%}，teacher KL={hard_kl:.5f}")  # 汇总硬标签基线指标
print(f"知识蒸馏：accuracy={distilled_accuracy:.1%}，teacher KL={distilled_kl:.5f}")  # 汇总蒸馏主方案指标

id | 标签 | 教师预测 | 硬学生 | 蒸馏学生 | KL硬 | KL蒸馏
K-01 | 普通 | 普通 | 普通 | 普通 | 0.0780 | 0.0525
K-02 | 加急 | 加急 | 加急 | 加急 | 0.3923 | 0.0724
K-03 | 加急 | 加急 | 加急 | 普通 | 0.0451 | 0.0836
K-04 | 严重 | 严重 | 严重 | 严重 | 0.0105 | 0.0021
K-05 | 普通 | 普通 | 普通 | 普通 | 0.0209 | 0.0061
K-06 | 严重 | 严重 | 严重 | 严重 | 0.0035 | 0.0027
K-07 | 加急 | 加急 | 加急 | 加急 | 0.0256 | 0.0056
K-08 | 严重 | 严重 | 严重 | 严重 | 0.0007 | 0.0028
K-09 | 普通 | 普通 | 普通 | 普通 | 0.0675 | 0.0442
K-10 | 加急 | 加急 | 加急 | 加急 | 0.1467 | 0.0061
硬标签：accuracy=100.0%，teacher KL=0.07909
知识蒸馏：accuracy=90.0%，teacher KL=0.02780


## 失败案例与修正：高温时遗漏 T²

温度升高会让 softmax 更平，logits 梯度大约缩小到 `1/T²`。下面在完全相同初始学生上分别计算梯度，验证补偿前后差异。

In [5]:
def soft_gradient_norm(temperature, compensate):  # 测量指定温度和补偿设置下的学生梯度
    probe = LinearStudent()  # 创建独立探针学生
    probe.load_state_dict(initial_student_state)  # 恢复相同初始化排除参数差异
    probe_logits = probe(x)  # 对同一十样本执行 forward
    probe_loss = soft_cross_entropy(probe_logits, teacher_logits, temperature, compensate)  # 计算指定软目标损失
    probe_loss.backward()  # 真实执行 backward 获取温度影响
    return float(torch.sqrt(sum(parameter.grad.square().sum() for parameter in probe.parameters())))  # 返回全局梯度范数
gradient_t1 = soft_gradient_norm(1.0, False)  # 测量常温未补偿梯度
gradient_t4_without = soft_gradient_norm(4.0, False)  # 复现高温遗漏 T² 的梯度缩小
gradient_t4_with = soft_gradient_norm(4.0, True)  # 测量加入 T² 后的高温梯度
print(f"T=1 无补偿 grad_norm={gradient_t1:.6f}")  # 展示常温参考梯度
print(f"T=4 无 T² grad_norm={gradient_t4_without:.6f}")  # 展示高温梯度过小失败
print(f"T=4 有 T² grad_norm={gradient_t4_with:.6f}")  # 展示补偿后的有效梯度
print(f"T² 补偿放大倍数={gradient_t4_with / gradient_t4_without:.1f}x")  # 验证理论上的十六倍补偿

T=1 无补偿 grad_norm=0.294568
T=4 无 T² grad_norm=0.027205
T=4 有 T² grad_norm=0.435281
T² 补偿放大倍数=16.0x


## 结果解读

硬标签学生可能也能命中类别，但它会趋向 one-hot，未必复现教师对相邻风险等级的犹豫。蒸馏结果用 teacher KL 显示分布级拟合改善。`T²` 只修正梯度尺度，不代表温度越高越好；温度和 alpha 都要通过验证集选择。

## 生产边界

本例教师是固定规则网络，学生只有线性层，没有真实 tokenizer、类别漂移或 teacher serving 成本。生产蒸馏应离线缓存教师 logits、版本和样本 ID，评估校准误差、长尾召回、延迟、显存及公平性。教师错误会被学生继承，不能把 teacher prediction 当作无条件真值。

## 最小回归测试

In [6]:
assert len(records) >= 5  # 保证案例包含多个可读业务样本
assert teacher_logits.shape == (len(records), len(label_names))  # 保证教师为每条样本输出完整类别分布
assert distill_trace[-1][3] < distill_trace[0][3]  # 保证真实蒸馏训练降低混合目标
assert distilled_kl < hard_kl  # 保证蒸馏学生比硬标签基线更接近教师分布
assert distilled_accuracy >= 0.8  # 保证学生在教学样本上保持基本分类能力
assert gradient_t4_with > gradient_t4_without * 15.9  # 保证 T² 对 T=4 梯度执行十六倍补偿
assert gradient_t1 > 0.0 and gradient_t4_with > 0.0  # 保证温度对照确实通过反向传播得到非零梯度